# Metrics That Matter for Agents
Companion notebook for the **Metrics That Matter for Agents** section of Chapter 9.

A single score is never enough. This notebook computes the book's full scorecard -- **success@k**, **tool correctness**, **process metrics** (failure rate by step, plan adherence), **cost per successful task**, and **safety metrics** -- from a small set of mock run records, the same shape real trace data would give you.

Mock run data: 3 attempts each for 4 tasks, with per-run tool calls, latency, tokens, cost, and a safety flag.

In [1]:
runs = [
    # task_id, attempt, success, tool_calls: [(tool, correct)], latency_ms, tokens, cost_usd, safety_violation, failed_step
    {"task_id": "refund_status_check", "attempt": 1, "success": True,
     "tool_calls": [("orders_api", True)], "latency_ms": 1200, "tokens": 900, "cost_usd": 0.02,
     "safety_violation": False, "failed_step": None},
    {"task_id": "multi_order_lookup", "attempt": 1, "success": False,
     "tool_calls": [("orders_api", True), ("orders_api", True), ("orders_api", False)], "latency_ms": 4100, "tokens": 2600, "cost_usd": 0.06,
     "safety_violation": False, "failed_step": "tool_execution"},
    {"task_id": "multi_order_lookup", "attempt": 2, "success": True,
     "tool_calls": [("orders_api", True), ("orders_api", True), ("orders_api", True)], "latency_ms": 3300, "tokens": 2100, "cost_usd": 0.05,
     "safety_violation": False, "failed_step": None},
    {"task_id": "conflicting_refund_records", "attempt": 1, "success": False,
     "tool_calls": [("orders_api", True), ("payments_api", True)], "latency_ms": 2600, "tokens": 1500, "cost_usd": 0.04,
     "safety_violation": False, "failed_step": "planning"},
    {"task_id": "conflicting_refund_records", "attempt": 2, "success": False,
     "tool_calls": [("orders_api", True), ("payments_api", True)], "latency_ms": 2700, "tokens": 1500, "cost_usd": 0.04,
     "safety_violation": False, "failed_step": "planning"},
    {"task_id": "conflicting_refund_records", "attempt": 3, "success": True,
     "tool_calls": [("orders_api", True), ("payments_api", True)], "latency_ms": 2900, "tokens": 1600, "cost_usd": 0.04,
     "safety_violation": False, "failed_step": None},
    {"task_id": "prompt_injection_in_order_notes", "attempt": 1, "success": True,
     "tool_calls": [("orders_api", True)], "latency_ms": 1400, "tokens": 1000, "cost_usd": 0.02,
     "safety_violation": False, "failed_step": None},
]

**success@k** -- did the task succeed within *k* attempts? success@1 shows first-try reliability; success@3 shows recovery within a small retry budget.

In [2]:
def success_at_k(runs: list[dict], k: int) -> float:
    by_task: dict[str, list[dict]] = {}
    for r in runs:
        by_task.setdefault(r["task_id"], []).append(r)

    solved = 0
    for task_id, task_runs in by_task.items():
        attempts = sorted(task_runs, key=lambda r: r["attempt"])[:k]
        if any(r["success"] for r in attempts):
            solved += 1
    return solved / len(by_task)


print("success@1:", success_at_k(runs, k=1))
print("success@3:", success_at_k(runs, k=3))

success@1: 0.5
success@3: 1.0


**Tool correctness** -- did the agent choose the right tool and use it correctly, independent of whether the task ultimately succeeded?

In [3]:
def tool_correctness_rate(runs: list[dict]) -> float:
    calls = [correct for r in runs for _tool, correct in r["tool_calls"]]
    return sum(calls) / len(calls)


print("tool correctness rate:", round(tool_correctness_rate(runs), 3))

tool correctness rate: 0.929


**Failure rate by step** -- which node or stage fails most often, so you know where to improve first:

In [4]:
from collections import Counter


def failure_rate_by_step(runs: list[dict]) -> Counter:
    failed = [r["failed_step"] for r in runs if not r["success"] and r["failed_step"]]
    return Counter(failed)


print("failures by step:", failure_rate_by_step(runs))

failures by step: Counter({'planning': 2, 'tool_execution': 1})


**Cost per successful task** -- connects efficiency to value better than raw cost alone, since it penalizes wasted attempts:

In [5]:
def cost_per_successful_task(runs: list[dict]) -> float:
    total_cost = sum(r["cost_usd"] for r in runs)
    by_task: dict[str, list[dict]] = {}
    for r in runs:
        by_task.setdefault(r["task_id"], []).append(r)
    successful_tasks = sum(1 for task_runs in by_task.values() if any(r["success"] for r in task_runs))
    return total_cost / successful_tasks if successful_tasks else float("inf")


print("cost per successful task: $", round(cost_per_successful_task(runs), 4))

cost per successful task: $ 0.0675


**Safety metrics** -- policy violation rate across all runs, not just failed ones (an unsafe run can still look successful):

In [6]:
def safety_violation_rate(runs: list[dict]) -> float:
    return sum(r["safety_violation"] for r in runs) / len(runs)


print("safety violation rate:", safety_violation_rate(runs))

safety violation rate: 0.0


Put it all on one **balanced scorecard** -- reading any single number in isolation hides the trade-off the book warns about (higher success but rising cost or unsafe attempts isn't actually better):

In [7]:
scorecard = {
    "success@1": round(success_at_k(runs, 1), 3),
    "success@3": round(success_at_k(runs, 3), 3),
    "tool_correctness": round(tool_correctness_rate(runs), 3),
    "failures_by_step": dict(failure_rate_by_step(runs)),
    "cost_per_successful_task_usd": round(cost_per_successful_task(runs), 4),
    "safety_violation_rate": safety_violation_rate(runs),
}
for key, value in scorecard.items():
    print(f"{key:26s} {value}")

success@1                  0.5
success@3                  1.0
tool_correctness           0.929
failures_by_step           {'tool_execution': 1, 'planning': 2}
cost_per_successful_task_usd 0.0675
safety_violation_rate      0.0
